# Proyecto Sprint 4 - Análisis de embudo y retención para MercadoLibre

**Introducción**

Eres analista de producto en MercadoLibre, dentro del equipo de Crecimiento y Retención.

El director de producto te plantea un reto:

_“Necesitamos entender en qué etapa del proceso perdemos usuarios y cómo podemos mejorar su retención a lo largo del tiempo.”_

Tu misión será usar SQL para mapear el embudo de conversión completo, identificar los principales puntos de fuga y evaluar la retención de usuarios por cohortes.

Finalmente, deberás proponer mejoras accionables basadas en los datos.

**Objetivos del proyecto**

Al finalizar este proyecto podrás:

- Construir embudos multietapa en SQL usando CTEs.
- Calcular tasas de conversión entre pasos y detectar caídas.
- Analizar la retención de usuarios por cohortes.
- Simular mejoras en conversión o retención.
- Validar resultados y comunicar hallazgos ejecutivos.

**Dataset del proyecto**

**Tablas disponibles:** Consultar en https://tripleten.com/trainer/data-analyst/lesson/bf5a7c10-9926-494c-a8b4-8b19767620c5/?from=program

**Contexto del negocio**

La dirección de producto busca responder:

**¿En qué etapa se pierden más usuarios?**

- Entre el [01/01/2025] y el [08/31/2025], ¿cuál es la tasa de conversión entre cada etapa clave del embudo?.
- ¿En qué paso se observa la mayor caída porcentual de usuarios?
- ¿Cómo varía esta pérdida por país (country)?

**Métricas clave:**

- Tasa de conversión por etapa
- Agrupación de eventos por device_category y referral_source

**¿Qué tan bien retenemos a los usuarios a lo largo del tiempo?**

- Para los usuarios que se registraron entre el [01/01/2025] y el [06/01/2025], ¿cuál es la tasa de retención en D7, D14, D21, D28?
- ¿Cómo se comporta la retención por país (country)?

**Métrica clave:** Retención por cohorte (D7, D14, D21, D28)

# Parte 1: Explorar el esquema y comprender el flujo

## **Paso 1:** Listar columnas y tipos de datos

**Objetivo:** conocer la estructura de la tabla `mercadolibre_funnel` 

**Instrucciones:**

- Imprime los primeros 5 renglones de la tabla.
- Identifica qué columnas te serán útiles para el análisis.

In [ ]:
SELECT *
FROM mercadolibre_funnel
LIMIT 5;

## **Paso 2:** Listar columnas y tipos de datos

**Objetivo:** conocer la estructura de la tabla `mercadolibre_retention.`

**Instrucciones:**

- Imprime los primeros 5 renglones de la tabla.
- Identifica qué columnas te serán útiles para el análisis.

In [ ]:
SELECT *
FROM mercadolibre_retention
LIMIT 5;

## **Paso 3:** Explorar tipos de eventos

**Objetivo:** confirmar la secuencia del embudo de la tabla `mercadolibre_funnel`

**Instrucciones:**

- Lista los eventos (sin duplicados)
- Ordena por el nombre del evento

In [ ]:
SELECT DISTINCT event_name
FROM mercadolibre_funnel
ORDER BY event_name;

# Parte 2: Construir el embudo de conversión

## **Paso 1:** Crear CTEs por etapa

**Objetivo:**

Construir bloques de usuarios únicos por evento (CTEs) en el rango 2025-01-01 → 2025-08-31, unirlos y contar usuarios por etapa del embudo.

Al unirlos, nos aseguramos de que todos los usuarios pasaron por cada etapa del embudo.

In [ ]:
WITH first_visit AS (
  SELECT DISTINCT user_id
  FROM mercadolibre_funnel
  WHERE event_name = 'first_visit'
    AND event_date BETWEEN '2025-01-01' AND '2025-08-31'
),
select_item AS (
  SELECT DISTINCT user_id
  FROM mercadolibre_funnel
  WHERE event_name IN ('select_item', 'select_promotion')
    AND event_date BETWEEN '2025-01-01' AND '2025-08-31'
),
add_to_cart AS (
  SELECT DISTINCT user_id
  FROM mercadolibre_funnel
  WHERE event_name = 'add_to_cart'
    AND event_date BETWEEN '2025-01-01' AND '2025-08-31'
),
begin_checkout AS (
  SELECT DISTINCT user_id
  FROM mercadolibre_funnel
  WHERE event_name = 'begin_checkout'
    AND event_date BETWEEN '2025-01-01' AND '2025-08-31'
),
add_shipping_info AS (
  SELECT DISTINCT user_id
  FROM mercadolibre_funnel
  WHERE event_name = 'add_shipping_info'
    AND event_date BETWEEN '2025-01-01' AND '2025-08-31'
),
add_payment_info AS (
  SELECT DISTINCT user_id
  FROM mercadolibre_funnel
  WHERE event_name = 'add_payment_info'
    AND event_date BETWEEN '2025-01-01' AND '2025-08-31'
),
purchase AS (
  SELECT DISTINCT user_id
  FROM mercadolibre_funnel
  WHERE event_name = 'purchase'
    AND event_date BETWEEN '2025-01-01' AND '2025-08-31'
)

SELECT
  COUNT(fv.user_id) AS usuarios_first_visit,
  COUNT(si.user_id) AS usuarios_select_item,
  COUNT(a.user_id) AS usuarios_add_to_cart,
  COUNT(bc.user_id) AS usuarios_begin_checkout,
  COUNT(asi.user_id) AS usuarios_add_shipping_info,
  COUNT(api.user_id) AS usuarios_add_payment_info,
  COUNT(p.user_id) AS usuarios_purchase
FROM first_visit fv
LEFT JOIN select_item si        ON fv.user_id = si.user_id
LEFT JOIN add_to_cart a         ON fv.user_id = a.user_id
LEFT JOIN begin_checkout bc     ON fv.user_id = bc.user_id
LEFT JOIN add_shipping_info asi ON fv.user_id = asi.user_id
LEFT JOIN add_payment_info api  ON fv.user_id = api.user_id
LEFT JOIN purchase p            ON fv.user_id = p.user_id;

## **Paso 2:** Calcular conversiones entre etapas

**Objetivo:** A partir de los conteos por etapa del embudo, calcular el porcentaje de conversión desde la etapa inicial (first_visit) hacia cada etapa.

**Instrucciones:**

Reutiliza el CTE `funnel_counts` (o la vista creada previamente) que ya tiene `usuarios_first_visit`, `usuarios_select_item`, `usuarios_add_to_cart`, etc.

1. En el SELECT final, crea 6 columnas de conversión, una por cada etapa posterior a `first_visit`, la etapa inicial.
    - Usa la fórmula `etapa actual * 100.0 / etapa inicial`.
    - Asigna los alias: `conversion_select_item`, `conversion_add_to_cart`, `conversion_begin_checkout`, `conversion_add_shipping_info`, `conversion_add_payment_info` y `conversion_purchase`.
2. Redondea a 2 decimales.
3. Usa `NULLIF(usuarios_first_visit, 0)` si temes división por cero.

In [ ]:
WITH first_visit AS (
  SELECT DISTINCT user_id
  FROM mercadolibre_funnel
  WHERE event_name = 'first_visit'
    AND event_date BETWEEN '2025-01-01' AND '2025-08-31'
),
select_item AS (
  SELECT DISTINCT user_id
  FROM mercadolibre_funnel
  WHERE event_name IN ('select_item', 'select_promotion')
    AND event_date BETWEEN '2025-01-01' AND '2025-08-31'
),
add_to_cart AS (
  SELECT DISTINCT user_id
  FROM mercadolibre_funnel
  WHERE event_name = 'add_to_cart'
    AND event_date BETWEEN '2025-01-01' AND '2025-08-31'
),
begin_checkout AS (
  SELECT DISTINCT user_id
  FROM mercadolibre_funnel
  WHERE event_name = 'begin_checkout'
    AND event_date BETWEEN '2025-01-01' AND '2025-08-31'
),
add_shipping_info AS (
  SELECT DISTINCT user_id
  FROM mercadolibre_funnel
  WHERE event_name = 'add_shipping_info'
    AND event_date BETWEEN '2025-01-01' AND '2025-08-31'
),
add_payment_info AS (
  SELECT DISTINCT user_id
  FROM mercadolibre_funnel
  WHERE event_name = 'add_payment_info'
    AND event_date BETWEEN '2025-01-01' AND '2025-08-31'
),
purchase AS (
  SELECT DISTINCT user_id
  FROM mercadolibre_funnel
  WHERE event_name = 'purchase'
    AND event_date BETWEEN '2025-01-01' AND '2025-08-31'
), 
funnel_counts AS(
SELECT
  COUNT(fv.user_id) AS usuarios_first_visit,
  COUNT(si.user_id) AS usuarios_select_item,
  COUNT(a.user_id) AS usuarios_add_to_cart,
  COUNT(bc.user_id) AS usuarios_begin_checkout,
  COUNT(asi.user_id) AS usuarios_add_shipping_info,
  COUNT(api.user_id) AS usuarios_add_payment_info,
  COUNT(p.user_id) AS usuarios_purchase
FROM first_visit fv
LEFT JOIN select_item si        ON fv.user_id = si.user_id
LEFT JOIN add_to_cart a         ON fv.user_id = a.user_id
LEFT JOIN begin_checkout bc     ON fv.user_id = bc.user_id
LEFT JOIN add_shipping_info asi ON fv.user_id = asi.user_id
LEFT JOIN add_payment_info api  ON fv.user_id = api.user_id
LEFT JOIN purchase p            ON fv.user_id = p.user_id
)

SELECT
ROUND(usuarios_select_item * 100.0/ usuarios_first_visit,2) AS conversion_select_item,
ROUND(usuarios_add_to_cart * 100.0/ usuarios_first_visit,2) AS conversion_add_to_cart,
ROUND(usuarios_begin_checkout * 100.0/ usuarios_first_visit,2) AS conversion_begin_checkout,
ROUND(usuarios_add_shipping_info * 100.0/ usuarios_first_visit,2) AS conversion_add_shipping_info,
ROUND(usuarios_add_payment_info * 100.0/ usuarios_first_visit,2) AS conversion_add_payment_info,
ROUND(usuarios_purchase * 100.0/ usuarios_first_visit,2) AS conversion_purchase
FROM funnel_counts;

## **Paso 3:** Segmentar el Embudo General por país

**Objetivo:** agrupa las conversiones del embudo por país (`country`) y detecta en que etapa del funnel se pierde más a los usuarios.

**Instrucciones**

1. Propaga el país en cada CTE del embudo:
    - Adicionalmente al usuario, selecciona el país (`country`) para poder acceder a la información. Mantén el rango de fechas.
2. En `funnel_counts`:
    - ajusta los LEFT JOINs para unir por usuario y país.
    - Cuenta usuarios únicos por etapa anclando siempre en `first_visits`
    - Recuerda agrupar por la columna que nos interesa.
3. Calcula las conversiones por país como % sobre los usuarios de `first_visits`:
    - `conversion_x = usuarios_x * 100.0/ usuarios_first_visits`
    - Usa los aliases pedidos en el precodigo.

**Devuelve una tabla** con el país y sus columnas de conversión. 
- Ordena por `conversion_purchase` de mayor a menor.

**Instrucciones paso a paso** 

Intenta realizando los ejercicios únicamente con las instrucciones de arriba, si necesitas más ayuda, lee las siguientes:

1. En cada CTE (`first_visits`, `select_item`, `add_to_cart`, `begin_checkout`, `add_shipping_info`, `add_payment_info`, `purchase`):
    - Selecciona `user_id` y `country`, evita los duplicados con `DISTINCT`.
    - Mantén el mismo `event_name` y el filtrado de fechas.
2. En el CTE `funnel_counts`:
    - Selecciona `fv.country` y cuenta `COUNT(DISTINCT <alias>.user_id)` para cada etapa.
    - Actualiza cada `LEFT JOIN` para agregar el match de país:
    - `... ON fv.user_id = <alias>.user_id AND fv.country = <alias>.country`.
    - Agrupa con `GROUP BY fv.country`.
3. En la consulta final:
    - Selecciona `country` y calcula:
        - `conversion_select_item`, `conversion_add_to_cart`, `conversion_begin_checkout`, `conversion_add_shipping_info`, `conversion_add_payment_info` y `conversion_purchase`.
    - Usa `usuarios_etapa * 100.0 / NULLIF(usuarios_first_visits , 0)` para evitar ÷ 0.
    - Ordena por `conversion_purchase DESC`.

In [ ]:
WITH first_visits AS (
  SELECT DISTINCT user_id, country
  FROM mercadolibre_funnel
  WHERE event_name = 'first_visit'
    AND event_date BETWEEN '2025-01-01' AND '2025-08-31'
),
select_item AS (
  SELECT DISTINCT user_id, country
  FROM mercadolibre_funnel
  WHERE event_name IN ('select_item', 'select_promotion')
    AND event_date BETWEEN '2025-01-01' AND '2025-08-31'
),
add_to_cart AS (
  SELECT DISTINCT user_id, country
  FROM mercadolibre_funnel
  WHERE event_name = 'add_to_cart'
    AND event_date BETWEEN '2025-01-01' AND '2025-08-31'
),
begin_checkout AS (
  SELECT DISTINCT user_id, country
  FROM mercadolibre_funnel
  WHERE event_name = 'begin_checkout'
    AND event_date BETWEEN '2025-01-01' AND '2025-08-31'
),
add_shipping_info AS (
  SELECT DISTINCT user_id, country
  FROM mercadolibre_funnel
  WHERE event_name = 'add_shipping_info'
    AND event_date BETWEEN '2025-01-01' AND '2025-08-31'
),
add_payment_info AS (
  SELECT DISTINCT user_id, country
  FROM mercadolibre_funnel
  WHERE event_name = 'add_payment_info'
    AND event_date BETWEEN '2025-01-01' AND '2025-08-31'
),
purchase AS (
  SELECT DISTINCT user_id, country
  FROM mercadolibre_funnel
  WHERE event_name = 'purchase'
    AND event_date BETWEEN '2025-01-01' AND '2025-08-31'
),
funnel_counts AS(
SELECT
  fv.country,
  COUNT(fv.user_id) AS usuarios_first_visit,
  COUNT(si.user_id) AS usuarios_select_item,
  COUNT(a.user_id) AS usuarios_add_to_cart,
  COUNT(bc.user_id) AS usuarios_begin_checkout,
  COUNT(asi.user_id) AS usuarios_add_shipping_info,
  COUNT(api.user_id) AS usuarios_add_payment_info,
  COUNT(p.user_id) AS usuarios_purchase
FROM first_visits fv
LEFT JOIN select_item si        ON fv.user_id = si.user_id AND fv.country = si.country
LEFT JOIN add_to_cart a         ON fv.user_id = a.user_id AND fv.country = a.country
LEFT JOIN begin_checkout bc     ON fv.user_id = bc.user_id AND fv.country = bc.country
LEFT JOIN add_shipping_info asi ON fv.user_id = asi.user_id AND fv.country = asi.country
LEFT JOIN add_payment_info api  ON fv.user_id = api.user_id AND fv.country = api.country
LEFT JOIN purchase p            ON fv.user_id = p.user_id AND fv.country = p.country
GROUP BY fv.country
)

-- Conversiones por país
SELECT
country, 
usuarios_select_item * 100.0 /NULLIF(usuarios_first_visit, 0) AS conversion_select_item,
usuarios_add_to_cart * 100.0 /NULLIF(usuarios_first_visit, 0) AS conversion_add_to_cart,
usuarios_begin_checkout * 100.0 /NULLIF(usuarios_first_visit, 0) AS conversion_begin_checkout,
usuarios_add_shipping_info * 100.0 /NULLIF(usuarios_first_visit, 0) AS conversion_add_shipping_info,
usuarios_add_payment_info * 100.0 /NULLIF(usuarios_first_visit, 0) AS conversion_add_payment_info,
usuarios_purchase * 100.0 /NULLIF(usuarios_first_visit, 0) AS conversion_purchase
FROM funnel_counts
ORDER BY conversion_purchase DESC;

# Parte 3 Analizar retención y cohortes

## **Paso 1:** Contar usuarios activos acomulados por país (D7, D14, D21, D28)

Objetivo: Para cada país, contar usuarios activos acumulados desde su registro, en el rango 2025-01-01 → 2025-08-31, al día 7, día 14, día 21 y día 28.

Instrucciones 

Cuenta usuarios activos acumulados con CASE. Usa los Alias users_d7, users_d14, users_d21, users_d28. Debes contar los usuarios que cumplan con estas dos condiciones:
usuarios que están activos (active = 1) .
usuarios donde day_after_signup es mayor o igual al día de interés.
Evita duplicados.
Filtra la tabla por activity_date entre '2025-01-01' y '2025-08-31'.
Agrupa y ordena por país country.
Instrucciones paso a paso (si las necesitas)

Intenta realizando los ejercicios únicamente con las instrucciones de arriba, si necesitas más ayuda, lee las siguientes:

Para cada hito (D7, D14, D21, D28) agrega una columna y usa COUNT con DISTINCT CASE WHEN para contar cuando:
day_after_signup sea mayor o igual a los días de interés. (e.j. day_after_signup ≥ 7)
el usuario sea activo, es decir active = 1
Aplica WHERE activity_date BETWEEN '2025-01-01' AND '2025-08-31'.
Recuerda el orden SELECT, FROM, WHERE, GROUP BY y ORDER BY.